# pipeline

> The headless transcription pipeline: VAD analysis → boundary computation → segment cutting → per-segment model-input conversion → transcription, composed over capability workers via the substrate's `JobQueue`.

Stage outputs are threaded into the next stage's inputs **manually** (run job → read result → submit next) because `submit_sequence` cannot pipe outputs to inputs (CR-16); this module is deliberately a real-world consumer of that gap — every workaround here is pass-2 evidence (see `claude-docs/pass-2-evidence.md`).

HITL approval seams use the cheapest viable form (log + optional CLI prompt) per the cores-cluster guard-rails; each seam carries its 5-field HITL-assist annotation in the docstring.

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue, JobStatus
from cjm_plugin_system.core.ports import (
    Composition, CompositionNode, CompositionRun, NodeState, OutputRef,
)
from cjm_plugin_system.core.empirical_store import compute_config_hash
from cjm_plugin_system.core.journal_store import JournalEvent, SubstrateEventType
from cjm_plugin_system.utils.hashing import hash_file

# Typed wire-kind registration (stage 2): importing the DTO classes is what
# lets the proxy's wire_decode hand this host process TYPED results.
from cjm_media_plugin_system.core import MediaAnalysisResult
from cjm_capability_primitives.transcription import TranscriptionResult

from cjm_transcription_core.models import (
    PipelineConfig,
    RunManifest,
    SegmentRecord,
    SourceResult,
    new_run_id,
)
from cjm_transcription_core.boundaries import compute_segment_boundaries
from cjm_transcription_core.emission import emit_source_graph

logger = logging.getLogger(__name__)

In [ ]:
#| export
async def submit_and_wait(
    queue: JobQueue,   # Started job queue
    instance_id: str,  # Capability instance to invoke
    *,
    timeout: Optional[float] = None,  # Seconds to wait; None = no limit
    **kwargs,          # Forwarded to the capability's execute()
) -> Any:  # The completed job's result payload
    """Submit one capability job, wait for it, and return its result (raise on failure)."""
    job_id = await queue.submit(instance_id, **kwargs)
    job = await queue.wait_for_job(job_id, timeout=timeout)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{instance_id} job {job_id} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
def normalize_vad_result(
    result: MediaAnalysisResult,  # Typed VAD result (wire-decoded at the proxy)
) -> Tuple[List[Dict[str, float]], float]:  # (sorted speech chunks [{start, end}], reported duration)
    """Normalize a typed VAD result into sorted speech chunks + the reported duration.

    Stage 2 (typed wire layer): the result arrives as a `MediaAnalysisResult`
    with typed `TimeRange` ranges — the dict-or-object tolerance (`field_of`,
    evidence E5) and the start/start_time key-variance handling retired with
    the untyped wire. Duration comes from the result metadata; returns 0.0
    when the capability did not report one (callers fall back to an ffmpeg
    probe).
    """
    chunks = [{"start": float(r.start), "end": float(r.end)} for r in result.ranges]
    chunks.sort(key=lambda c: c["start"])
    duration = float((result.metadata or {}).get("duration", 0.0) or 0.0)
    return chunks, duration

In [ ]:
#| export
async def analyze_vad(
    queue: JobQueue,
    vad_id: str,          # VAD capability instance id
    audio_path: str,      # Audio file to analyze
    force: bool = False,  # Bypass the VAD capability's (path, config_hash) cache
) -> Tuple[List[Dict[str, float]], float]:  # (speech chunks, reported duration)
    """Run VAD analysis on one audio file."""
    result = await submit_and_wait(queue, vad_id, media_path=audio_path, force=force)
    return normalize_vad_result(result)

In [ ]:
#| export
async def probe_duration(
    queue: JobQueue,
    ffmpeg_id: str,   # ffmpeg capability instance id
    audio_path: str,  # Audio file to probe
) -> float:  # Duration in seconds (0.0 when the probe fails to report one)
    """Probe a media file's duration via the ffmpeg capability's `get_info` action."""
    info = await submit_and_wait(queue, ffmpeg_id, action="get_info", file_path=audio_path)
    # ffmpeg action results are plugin-side dicts (E6) — plain dict access
    # (the multi-op tool stays untyped until its stage-8 adapter split).
    return float((info or {}).get("duration", 0.0) or 0.0)

In [ ]:
#| export
async def cut_segments(
    queue: JobQueue,
    ffmpeg_id: str,                      # ffmpeg capability instance id
    audio_path: str,                     # Source audio to cut
    boundaries: List[Dict[str, float]],  # [{start, end}, ...] from compute_segment_boundaries
) -> Tuple[List[Dict[str, Any]], str]:  # (per-segment dicts from ffmpeg, batch_key)
    """Cut the source audio at the computed boundaries via ffmpeg `segment_audio`."""
    result = await submit_and_wait(
        queue, ffmpeg_id,
        action="segment_audio", input_path=audio_path, boundaries=boundaries,
    )
    segments = list((result or {}).get("segments") or [])
    batch_key = str((result or {}).get("batch_key") or "")
    if not segments:
        raise RuntimeError(f"segment_audio produced no segments for {audio_path}: {result!r}")
    return segments, batch_key

In [ ]:
#| export
def build_segment_composition(
    raw_segments: List[Dict[str, Any]],  # Per-segment dicts from ffmpeg segment_audio
    run_id: str,           # Run id (prefixes per-segment provenance job ids)
    source_index: int,     # Position of this source within the run
    ffmpeg_id: str,        # ffmpeg capability instance id
    transcriber_ids: List[str],  # Transcription capability instance ids (one or more)
    sample_rate: int = 16000,  # Model-input sample rate
    channels: int = 1,         # Model-input channel count
    force: bool = False,       # Per-call cache-bypass control flag
) -> Tuple[Composition, List[Dict[str, Any]]]:  # (composition, per-segment meta rows)
    """Build the per-source fan-out composition: N independent convert→(T× transcribe) pipes.

    Host-constructed fan-out (stage-3 ratified shape): the host computes every
    per-item kwarg statically; the only execution-time unknown, ffmpeg's hashed
    `cache_dir_for_config` output path, flows through the `OutputRef` binding.
    Stage 5 (dual-transcriber = the named parallel-port adopter): each segment's
    convert output fans out to ONE transcribe node PER TRANSCRIBER — the
    lightweight ∥ accuracy comparison runs as one composition; the transcribe
    nodes are independent and parallelize under the queue's empirical admission.
    """
    nodes: List[CompositionNode] = []
    metas: List[Dict[str, Any]] = []
    for seg in raw_segments:
        # ffmpeg per-segment entries are plugin-side dicts (untyped until the
        # multi-op tool's stage-8 adapter split).
        idx = int(seg.get("index", len(metas)))
        seg_path = str(seg.get("output_path", ""))
        start = float(seg.get("start", 0.0))
        end = float(seg.get("end", 0.0))
        conv = f"convert_{idx:04d}"
        nodes.append(CompositionNode(conv, ffmpeg_id, {
            "action": "convert", "input_path": seg_path,
            "output_format": "wav", "sample_rate": sample_rate, "channels": channels,
        }))
        transcribe_nodes: Dict[str, str] = {}
        job_ids: Dict[str, str] = {}
        for ti, transcriber_id in enumerate(transcriber_ids):
            tr = f"transcribe_t{ti}_{idx:04d}"
            job_id = f"{run_id}_src{source_index}_seg{idx:04d}_t{ti}"
            nodes.append(CompositionNode(tr, transcriber_id, {
                "audio": OutputRef(conv, "output_path"),
                # job_id / source_*_time ride the CR-15 identity/provenance kwarg
                # channel; force is the per-call control flag (CR-15 category 4).
                "job_id": job_id,
                "source_start_time": start,
                "source_end_time": end,
            }, task_name="transcription", method="transcribe", control={"force": force}))
            transcribe_nodes[transcriber_id] = tr
            job_ids[transcriber_id] = job_id
        metas.append({"index": idx, "segment_path": seg_path, "start": start,
                      "end": end, "job_ids": job_ids, "convert_node": conv,
                      "transcribe_nodes": transcribe_nodes})
    return Composition(nodes=nodes), metas

In [ ]:
#| export
def records_from_composition(
    crun: CompositionRun,          # Terminal composition run
    metas: List[Dict[str, Any]],   # Meta rows from build_segment_composition
) -> List[SegmentRecord]:  # Ordered per-segment records
    """Fold a completed segment composition back into SegmentRecords.

    Raises on a non-completed run, surfacing the failed nodes' structured
    errors — under fail_fast a single segment failure stops the source,
    matching the pre-ports loop where the first raise aborted the source.
    Stage 5: each record carries per-transcriber `transcripts` (symmetric
    variants; authority is the decomp consumer's choice).
    """
    if crun.status != NodeState.completed:
        failed = {nid: str(nr.error) for nid, nr in crun.node_runs.items()
                  if nr.state == NodeState.failed}
        raise RuntimeError(f"segment composition {crun.status.value}: {failed}")
    results = crun.results_by_node()
    records: List[SegmentRecord] = []
    for m in metas:
        wav_path = str((results[m["convert_node"]] or {}).get("output_path") or "")
        transcripts: Dict[str, Dict[str, Any]] = {}
        for transcriber_id, node_id in m["transcribe_nodes"].items():
            tr = results[node_id]
            # Typed TranscriptionResult (stage-2 wire layer): attribute access.
            transcripts[transcriber_id] = {
                "job_id": m["job_ids"][transcriber_id],
                "text": str(tr.text or ""),
                "metadata": dict(tr.metadata or {}),
            }
        records.append(SegmentRecord(
            index=m["index"], start=m["start"], end=m["end"],
            duration=m["end"] - m["start"],
            segment_path=m["segment_path"], model_input_path=wav_path,
            transcripts=transcripts,
        ))
    return records

In [ ]:
#| export
def tier1_segment_checks(
    boundaries: List[Dict[str, float]],  # Computed segment boundaries
    max_segment_duration: float,         # The configured wall-clock cap
    chunk_count: int,                    # VAD speech-chunk count
) -> List[str]:  # Human-readable warnings (empty = all clear)
    """Tier-1 deterministic pre-filters for the boundary-review seam (no AI)."""
    warnings: List[str] = []
    if chunk_count == 0:
        warnings.append("VAD detected NO speech chunks — source may be silent or non-speech")
    for i, b in enumerate(boundaries[:-1]):
        if (b["end"] - b["start"]) > max_segment_duration:
            warnings.append(
                f"non-final segment {i} exceeds max duration: {b['end'] - b['start']:.1f}s"
            )
    if boundaries:
        final = boundaries[-1]
        if (final["end"] - final["start"]) > 2 * max_segment_duration:
            warnings.append(
                f"final segment unusually long ({final['end'] - final['start']:.1f}s) — long trailing silence?"
            )
    return warnings

In [ ]:
#| export
def tier1_transcript_checks(
    segments: List[SegmentRecord],  # Transcribed segments for one source
) -> List[str]:  # Human-readable warnings (empty = all clear)
    """Tier-1 deterministic pre-filters for the transcript-review seam (no AI)."""
    warnings: List[str] = []
    for s in segments:
        for tname, tr in s.transcripts.items():
            text = str(tr.get("text") or "")
            if not text.strip():
                warnings.append(
                    f"segment {s.index} [{tname}] produced EMPTY text ({s.duration:.1f}s of audio)"
                )
            elif s.duration > 30 and len(text) < 20:
                warnings.append(
                    f"segment {s.index} [{tname}]: suspiciously short text ({len(text)} chars for {s.duration:.1f}s)"
                )
    return warnings

In [ ]:
#| export
def confirm_seam(
    seam: str,                 # Seam label, e.g. "boundary-review"
    summary_lines: List[str],  # What the operator is being asked to accept
    warnings: List[str],       # Tier-1 warnings (logged prominently)
    assume_yes: bool = False,  # Headless mode: accept without prompting
) -> bool:  # True = proceed, False = operator aborted
    """HITL approval seam in its cheapest viable form (log + optional CLI prompt).

    Per-seam capability annotation (HITL-assist methodology, 5 fields):
      1. signal: per-source summaries + Tier-1 warnings
      2. deterministic pre-filter: the tier1_* check functions (no AI)
      3. modality-bridge candidate: spectrogram render for boundary sanity (future Tier 2)
      4. authoritative verifier: re-transcribe-and-compare via a second capability (future Tier 3)
      5. flywheel capture: accept/abort decisions are logged; durable capture is
         a pass-2 seam-contract concern, not solved here

    NOTE: input() blocks the event loop — acceptable because seams sit between
    stages with no jobs in flight; the pass-2 seam contract needs an async shape.
    """
    for line in summary_lines:
        logger.info(f"[{seam}] {line}")
    for w in warnings:
        logger.warning(f"[{seam}] {w}")
    if assume_yes:
        logger.info(f"[{seam}] auto-accepted (assume_yes)")
        return True
    reply = input(f"[{seam}] proceed? [Y/n] ").strip().lower()
    accepted = reply in ("", "y", "yes")
    logger.info(f"[{seam}] {'accepted' if accepted else 'ABORTED'} by operator")
    return accepted

In [ ]:
#| export
async def run_source(
    queue: JobQueue,
    cfg: PipelineConfig,  # Run configuration
    source_path: str,     # Source audio file
    run_id: str,          # Run id (prefixes per-segment job ids)
    source_index: int,    # Position of this source within the run
) -> Optional[SourceResult]:  # None when the operator aborts at a seam
    """Run the full pipeline for one source: VAD → boundaries → cut → convert → transcribe."""
    t0 = time.time()
    logger.info(f"[src {source_index}] {source_path}")

    # 0. Content-address the source (the Source node identity input; stage 5).
    content_hash = hash_file(source_path)

    # 1. VAD analysis (duration falls back to an ffmpeg probe when unreported)
    chunks, duration = await analyze_vad(queue, cfg.vad_plugin, source_path, force=cfg.force)
    if duration <= 0:
        duration = await probe_duration(queue, cfg.ffmpeg_plugin, source_path)
    logger.info(f"[src {source_index}] VAD: {len(chunks)} speech chunks over {duration:.1f}s")

    # 2. Boundaries (pure logic)
    boundaries = compute_segment_boundaries(chunks, cfg.max_segment_duration, duration)

    # 3. HITL seam: boundary review
    if boundaries:
        longest = max(b["end"] - b["start"] for b in boundaries)
        summary = [f"{Path(source_path).name}: {len(boundaries)} segment(s), longest {longest:.1f}s"]
    else:
        summary = [f"{Path(source_path).name}: no segments computed"]
    if not confirm_seam(
        "boundary-review", summary,
        tier1_segment_checks(boundaries, cfg.max_segment_duration, len(chunks)),
        assume_yes=cfg.assume_yes,
    ):
        return None

    # 4. Cut the source at the boundaries
    raw_segments, batch_key = await cut_segments(queue, cfg.ffmpeg_plugin, source_path, boundaries)

    # 5. Per segment: convert → (T× transcribe) as ONE composition of N
    # independent pipes (CR-16 ports; stage-5 dual-transcriber fan-out).
    comp, metas = build_segment_composition(
        raw_segments, run_id, source_index,
        cfg.ffmpeg_plugin, cfg.transcriber_plugins,
        sample_rate=cfg.sample_rate, channels=cfg.channels, force=cfg.force,
    )
    comp_id = await queue.submit_composition(comp)
    crun = await queue.wait_for_composition(comp_id)
    records = records_from_composition(crun, metas)

    # 5b. Content-address the model-input WAVs (AudioSegment provenance; the
    # audio of record, E14 — graph emission + downstream slice refs hang off it).
    for r in records:
        if r.model_input_path:
            r.model_input_hash = hash_file(r.model_input_path)
        for tname, tr in r.transcripts.items():
            logger.info(f"[src {source_index}] seg {r.index} [{tname}]: {len(tr.get('text') or '')} chars")

    # 6. HITL seam: transcript review
    total_chars = {t: sum(len(r.transcripts.get(t, {}).get("text") or "") for r in records)
                   for t in cfg.transcriber_plugins}
    chars_summary = "  ".join(f"{t}: {n} chars" for t, n in total_chars.items())
    if not confirm_seam(
        "transcript-review",
        [f"{Path(source_path).name}: {len(records)} segment(s)  {chars_summary}"],
        tier1_transcript_checks(records),
        assume_yes=cfg.assume_yes,
    ):
        return None

    logger.info(f"[src {source_index}] done in {time.time() - t0:.1f}s")
    return SourceResult(
        source_path=source_path, duration=duration,
        vad_chunk_count=len(chunks), batch_key=batch_key,
        content_hash=content_hash, segments=records,
    )

In [ ]:
#| export
def collect_plugin_info(
    manager: PluginManager,   # Manager holding the loaded capabilities
    instance_ids: List[str],  # Instance ids to record
) -> Dict[str, Dict[str, Any]]:  # instance_id -> {name, version, db_path, config_hash}
    """Record capability identity + data-DB pointers for the run manifest (provenance).

    Stage 5: also records each capability's EFFECTIVE config hash (the same
    `compute_config_hash` the empirical store keys on) — Transcript node
    identity is (audio segment, transcriber, config_hash), so the manifest must
    carry the hash for downstream id recomputation. `db_path` prefers the
    effective config over the manifest default (the D19 lesson). Stage 6 (0.2.1): the EFFECTIVE config
    dict is recorded READABLY beside its hash -- the I8 lesson (a persisted
    stress config was only diagnosable by hash archaeology; bundle recipients
    should read model identity directly).
    """
    info: Dict[str, Dict[str, Any]] = {}
    for iid in instance_ids:
        meta = (getattr(manager, "plugins", {}) or {}).get(iid)
        if meta is None:
            continue
        manifest = getattr(meta, "manifest", {}) or {}
        current_config: Dict[str, Any] = {}
        try:
            proxy = manager.get_plugin(iid)
            if proxy is not None:
                current_config = proxy.get_current_config() or {}
        except Exception as e:  # Best-effort: identity recording must not fail the run
            logger.warning(f"collect_plugin_info: get_current_config({iid}) failed: {e}")
        info[iid] = {
            "name": meta.name,
            "version": getattr(meta, "version", None),
            "db_path": current_config.get("db_path") or manifest.get("db_path"),
            "config_hash": compute_config_hash(current_config),
            "config": current_config,
        }
    return info

In [ ]:
#| export
def _journal_run_event(
    manager: PluginManager,  # Manager owning the journal store
    event_type: str,         # SubstrateEventType value (run_started / run_finished)
    run_id: str,             # This run's manifest id
    actor: Optional[str],    # Who/what initiated the run
    payload: Dict[str, Any], # Run-level structured detail
) -> None:
    """Append a host-tier run event to the journal (CR-14 follow-up).

    The cores are the trusted host writer class: RUN_STARTED/RUN_FINISHED
    bracket the run so the run manifest (same run_id) links to every job row
    the run produced. No-op when the manager has no journal store (test
    doubles); append failures stay LOUD (journal contract).
    """
    journal = getattr(manager, "journal_store", None)
    if journal is None:
        return
    journal.append(JournalEvent(
        event_type=event_type, run_id=run_id, actor=actor, payload=payload))

In [ ]:
#| export
async def run_pipeline(
    manager: PluginManager,  # Manager with the capabilities loaded
    queue: JobQueue,         # Started job queue
    cfg: PipelineConfig,     # Run configuration
    sources: List[str],      # Source audio paths, in order
    run_id: Optional[str] = None,  # Override run id (default: generated)
    actor: Optional[str] = None,   # Who/what initiated (journal attribution; CLI default cli:<user>)
) -> RunManifest:  # Manifest of everything the run produced
    """Run the transcription pipeline over the given sources, in order.

    An operator abort at any seam stops the run; the manifest holds the sources
    completed so far (capability-side caches make re-runs cheap). With
    `cfg.graph_plugin` set, each completed source EMITS the graph root
    (Source → AudioSegment → Transcript; CR-18 revolution 2) idempotently —
    re-runs verify-collide instead of duplicating.
    """
    run_id = run_id or new_run_id()
    # CR-14 follow-up: queue-scoped run context — every job submitted in this
    # run carries run_id/actor into its journal rows + worker diagnostics
    # (run-manifest <-> journal linkage); the run itself is bracketed by
    # RUN_STARTED/RUN_FINISHED host-tier rows.
    queue.set_run_context(run_id=run_id, actor=actor)
    _journal_run_event(manager, SubstrateEventType.RUN_STARTED.value, run_id, actor, {
        "core": "cjm-transcription-core",
        "sources": [str(s) for s in sources],
        "transcribers": list(cfg.transcriber_plugins),
        "graph_plugin": cfg.graph_plugin,
    })
    plugin_ids = ([cfg.vad_plugin, cfg.ffmpeg_plugin] + list(cfg.transcriber_plugins)
                  + ([cfg.graph_plugin] if cfg.graph_plugin else []))
    manifest = RunManifest(
        run_id=run_id,
        created_at=time.time(),
        config=cfg.to_dict(),
        plugins=collect_plugin_info(manager, plugin_ids),
    )
    if cfg.graph_plugin:
        manifest.graph = {
            "plugin": cfg.graph_plugin,
            "db_path": (manifest.plugins.get(cfg.graph_plugin) or {}).get("db_path"),
        }
    transcriber_config_hashes = {
        t: str((manifest.plugins.get(t) or {}).get("config_hash") or "")
        for t in cfg.transcriber_plugins
    }
    status = "completed"
    try:
        for i, src in enumerate(sources):
            result = await run_source(queue, cfg, str(src), run_id, i)
            if result is None:
                logger.warning(
                    f"run {run_id}: aborted at source {i} ({src}); manifest holds {i} source(s)"
                )
                status = "aborted"
                break
            if cfg.graph_plugin:
                result.graph = await emit_source_graph(
                    queue, cfg.graph_plugin, result, transcriber_config_hashes, run_id,
                )
                logger.info(f"[src {i}] graph emission: {result.graph}")
            manifest.sources.append(result)
    except BaseException as e:
        # The journal exists for exactly this row: a run that DIED records
        # how far it got (failures stop being the unattributed case).
        _journal_run_event(manager, SubstrateEventType.RUN_FINISHED.value, run_id, actor, {
            "core": "cjm-transcription-core", "status": "failed", "error": repr(e),
            "sources_completed": len(manifest.sources), "sources_total": len(sources),
        })
        raise
    _journal_run_event(manager, SubstrateEventType.RUN_FINISHED.value, run_id, actor, {
        "core": "cjm-transcription-core", "status": status,
        "sources_completed": len(manifest.sources), "sources_total": len(sources),
        "segments": sum(len(s.segments) for s in manifest.sources),
    })
    return manifest

In [ ]:
# Pure-logic smoke checks (no plugins involved)
from cjm_media_plugin_system.core import TimeRange

# normalize_vad_result consumes the TYPED result (stage-2 wire layer);
# ordering still normalizes, duration still reads from metadata.
chunks, dur = normalize_vad_result(MediaAnalysisResult(
    ranges=[TimeRange(start=5.0, end=9.0), TimeRange(start=0.5, end=2.0)],
    metadata={"duration": 28.0},
))
assert chunks == [{"start": 0.5, "end": 2.0}, {"start": 5.0, "end": 9.0}], chunks
assert dur == 28.0

# tier-1 checks fire on the right shapes (0.2.0 per-transcriber transcripts)
assert tier1_segment_checks([], 300.0, 0) != []
assert tier1_segment_checks([{"start": 0.0, "end": 28.0}], 300.0, 3) == []
_empty = SegmentRecord(0, 0.0, 40.0, 40.0, "a", "b",
                       transcripts={"whisper": {"job_id": "j", "text": "", "metadata": {}}})
_ok = SegmentRecord(0, 0.0, 28.0, 28.0, "a", "b",
                    transcripts={"whisper": {"job_id": "j", "text": "plenty of text here", "metadata": {}}})
assert tier1_transcript_checks([_empty]) != []
assert tier1_transcript_checks([_ok]) == []
# per-transcriber warnings name the transcriber
_dual = SegmentRecord(0, 0.0, 40.0, 40.0, "a", "b", transcripts={
    "whisper": {"job_id": "j1", "text": "plenty of text here too", "metadata": {}},
    "voxtral": {"job_id": "j2", "text": "", "metadata": {}},
})
ws = tier1_transcript_checks([_dual])
assert len(ws) == 1 and "[voxtral]" in ws[0]

# confirm_seam headless path
assert confirm_seam("boundary-review", ["x"], [], assume_yes=True) is True
print("pipeline pure-logic checks OK")

In [ ]:
# Composition builder + folder pure-logic checks (no plugins involved)
from cjm_capability_primitives.transcription import TranscriptionResult
from cjm_plugin_system.core.ports import new_composition_run

_raw = [
    {"index": 0, "output_path": "/seg0.flac", "start": 0.0, "end": 280.0},
    {"index": 1, "output_path": "/seg1.flac", "start": 280.0, "end": 560.0},
]
_comp, _metas = build_segment_composition(_raw, "runX", 0, "ffmpeg", ["whisper"])
assert len(_comp.nodes) == 4 and len(_metas) == 2
# Each pipe: transcribe bound to its OWN convert's output_path.
assert _comp.nodes[1].kwargs["audio"] == OutputRef("convert_0000", "output_path")
assert _comp.nodes[3].kwargs["audio"] == OutputRef("convert_0001", "output_path")
# Per-item provenance kwargs are STATIC (host-computed at construction).
assert _comp.nodes[1].kwargs["job_id"] == "runX_src0_seg0000_t0"
assert _comp.nodes[3].kwargs["source_start_time"] == 280.0
# The two pipes are independent: converts have no deps; fan-out parallelizes.
_run = new_composition_run(_comp, "r")
assert _run.ready_nodes() == ["convert_0000", "convert_0001"]

# Fold a completed run back into records (per-transcriber transcripts).
_run.record_result("convert_0000", NodeState.completed, result={"output_path": "/c0.wav"})
_run.record_result("convert_0001", NodeState.completed, result={"output_path": "/c1.wav"})
_run.record_result("transcribe_t0_0000", NodeState.completed,
                   result=TranscriptionResult(text="hello", metadata={"m": 1}))
_run.record_result("transcribe_t0_0001", NodeState.completed,
                   result=TranscriptionResult(text="world", metadata={}))
_run.status = NodeState.completed
_recs = records_from_composition(_run, _metas)
assert [r.transcripts["whisper"]["text"] for r in _recs] == ["hello", "world"]
assert _recs[0].model_input_path == "/c0.wav"
assert _recs[1].transcripts["whisper"]["job_id"] == "runX_src0_seg0001_t0"
assert _recs[0].duration == 280.0

# DUAL-transcriber fan-out (stage 5: the named parallel-port adopter): one
# convert per segment, one transcribe node per transcriber off the same output.
_comp2, _metas2 = build_segment_composition(_raw, "runX", 0, "ffmpeg", ["whisper", "voxtral"])
assert len(_comp2.nodes) == 6
assert _comp2.nodes[1].kwargs["audio"] == _comp2.nodes[2].kwargs["audio"] == OutputRef("convert_0000", "output_path")
assert _comp2.nodes[1].kwargs["job_id"] == "runX_src0_seg0000_t0"
assert _comp2.nodes[2].kwargs["job_id"] == "runX_src0_seg0000_t1"
assert _metas2[0]["transcribe_nodes"] == {"whisper": "transcribe_t0_0000", "voxtral": "transcribe_t1_0000"}
_run2 = new_composition_run(_comp2, "r2")
assert _run2.ready_nodes() == ["convert_0000", "convert_0001"]
_run2.record_result("convert_0000", NodeState.completed, result={"output_path": "/c0.wav"})
_run2.record_result("convert_0001", NodeState.completed, result={"output_path": "/c1.wav"})
for nid, txt in [("transcribe_t0_0000", "w0"), ("transcribe_t1_0000", "v0"),
                 ("transcribe_t0_0001", "w1"), ("transcribe_t1_0001", "v1")]:
    _run2.record_result(nid, NodeState.completed, result=TranscriptionResult(text=txt, metadata={}))
_run2.status = NodeState.completed
_recs2 = records_from_composition(_run2, _metas2)
assert _recs2[0].transcripts["whisper"]["text"] == "w0" and _recs2[0].transcripts["voxtral"]["text"] == "v0"
assert _recs2[1].transcripts["voxtral"]["text"] == "v1"

# Non-completed run raises with the failed nodes surfaced.
_bad = new_composition_run(_comp, "r3")
_bad.record_result("convert_0000", NodeState.failed)
_bad.status = NodeState.failed
try:
    records_from_composition(_bad, _metas)
    raise AssertionError("expected failure surfacing")
except RuntimeError as e:
    assert "convert_0000" in str(e)
print("composition builder/folder checks OK")